In [29]:
import pandas as pd
import numpy as np
import duckdb
from collections import defaultdict
from tqdm import tqdm

import xml.etree.ElementTree as ET
from collections import defaultdict
import ast
import re

import sys, os
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.append(project_root)

# from modules.graph_construction.graph_snapshot import *
# from modules.graph_construction.enrich.mesh_desc import *
# from modules.dataset_preprocessing.enrichment import *
from shared_functions.global_functions import *
# from shared_functions.gg_sheet_drive import *
from modules.dataset_preprocessing.utils import *

#### Check Current stats

In [52]:
query_neo4j('''
match (n:Test)
return count(n)
''')

[{'count(n)': 238660}]

In [53]:
query_neo4j('''
match (n:Test)-[r]->(m:Test)
return count(r)
''')

[{'count(r)': 499346}]

In [51]:
query_neo4j('''
MATCH (n:Test)
RETURN labels(n) AS label, count(n) AS count
ORDER BY count DESC
''')

[{'label': ['Test', 'Drug', 'CTD'], 'count': 165053},
 {'label': ['Disease', 'HPO', 'Test'], 'count': 19941},
 {'label': ['Test', 'DB', 'Drug'], 'count': 19782},
 {'label': ['Disease', 'Test', 'CTD'], 'count': 11690},
 {'label': ['Test', 'Drug', 'PubMed'], 'count': 10164},
 {'label': ['Disease', 'Test', 'DO'], 'count': 8193},
 {'label': ['Disease', 'Test', 'PubChem'], 'count': 2626},
 {'label': ['Disease', 'Test', 'PubMed'], 'count': 1000},
 {'label': ['Test', 'Drug', 'PubChem'], 'count': 211}]

In [168]:
query_neo4j('''
MATCH ()-[r]->()
RETURN type(r) AS relationship, count(r) AS count
ORDER BY count DESC
''')

[{'relationship': 'CHILD_OF', 'count': 279015},
 {'relationship': 'CAUSE', 'count': 108867},
 {'relationship': 'INTERACTS_WITH', 'count': 80368},
 {'relationship': 'TREAT', 'count': 26315}]

#### Path Definition

In [2]:
data_dir = 'F:/Din/Study/Education/Projects/Thesis/data'

base_data_dir = os.path.join(project_root, 'data')
edge_dir = os.path.join(base_data_dir, 'edges.csv')
node_dir = os.path.join(base_data_dir, 'nodes.csv')

# MIMIC-IV
mimic_path = os.path.join(data_dir, 'mimic_iv')

hosp_path = os.path.join(mimic_path, 'hosp')

icu_path = os.path.join(mimic_path, 'icu')

note_path = os.path.join(mimic_path, 'note')

# BCD5CDR
bc5cdr_path = os.path.join(data_dir, 'bc5cdr', 'data', 'training')

# ChemDisGene
chemdisgene_path = os.path.join(data_dir, 'ChemDisGene', 'data', 'ctd_derived')

# DrugBank
drugbank_path = os.path.join(data_dir, 'DrugBank')

# CTD
ctd_path = os.path.join(data_dir, 'CTD')

# LOINC
loinc_path = os.path.join(data_dir, 'LOINC')

# SIDER
sider_path = os.path.join(data_dir, 'SIDER')

# FAERS
faers_path = os.path.join(data_dir, 'FAERS', 'XML')

# Snap
snap_path = os.path.join(data_dir, 'snap_drug_interaction')

# SnomedCT
snomedct_path = os.path.join(data_dir, 'SnomedCT', 'csv')

# RxNorm
rxnorm_path = os.path.join(data_dir, 'RxNorm')

# HPO
hpo_path = os.path.join(data_dir, 'HPO')

# DOID
doid_path = os.path.join(data_dir, 'DOID')

# UML
uml_path = os.path.join(data_dir, 'UML', 'META')

#### Test

In [3]:
base_data_dir = os.path.join(project_root, 'data')
edge_dir = os.path.join(base_data_dir, 'edges.csv')
node_dir = os.path.join(base_data_dir, 'nodes.csv')

nodes = pd.read_csv(node_dir)

D:\temp\ipykernel_16020\1700699268.py:5: DtypeWarning: Columns (9) have mixed types. Specify dtype option on import or set low_memory=False.
  nodes = pd.read_csv(node_dir)


In [ ]:
nodes = nodes[~nodes['mesh_id'].isna()]

nodes = nodes[nodes['description'].isna()]

nodes = nodes[~nodes['mesh_id'].str.startswith('DB')]

nodes = nodes.dropna(subset = 'description')

In [11]:
nodes

,id,labels,alias,description,doid_id,mesh_id,name,omim_id,pubchem_id,uml_id
0,HP:0002497,Disease:HPO:Test,[],A genetically heterogeneous group of hereditar...,['0050952'],C564815,Spastic Ataxia,[],NaN,NaN
1,HP:0003040,Disease:HPO:Test,['Disease of the joints'],Diseases involving the JOINTS.,[],D007592,Arthropathy,[],NaN,C0702154
2,HP:0006557,Disease:HPO:Test,[],A predominantly autosomal dominant hereditary ...,['0050770'],C536330,Polycystic Liver Disease,"['174050', '617004']",NaN,NaN
3,HP:0006789,Disease:HPO:Test,[],Mitochondrial myopathy encephalopathy lactic a...,[],C538525,Mitochondrial Encephalopathy,[],NaN,NaN
5,HP:0007583,Disease:HPO:Test,[],A very rare skin disease. Some researchers con...,[],C000715747,Telangiectasia Macularis Eruptiva Perstans,[],NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
179152,C536611,Disease:Test:CTD,"['Ancell-Spiegler Cylindromas', 'Brooke-Fordyc...","Familial cylindromatosis (OMIM: 132700), Brook...",[],C536611,Familial Cylindromatosis,"['132700', '601606', '605041']",NaN,NaN
179153,C535720,Disease:Test:CTD,"['Duodenal Atresia', 'Duodenal Stenosis']",A congenital absence or complete closure of th...,['0080216'],C535720,Familial Duodenal Atresia,[],NaN,NaN
179161,C536911,Disease:Test:CTD,"['Fmtc', 'Medullary Thyroid Cancer, Familial',...",A malignant tumor of the CALCITONIN - secretin...,['0050547'],C536911,Familial Medullary Thyroid Carcinoma,['155240'],NaN,NaN
179165,C536847,Disease:Test:CTD,"['Hereditary Multiple Trichodiscomas', 'Tricho...",Small benign fibrovascular tumor of the dermal...,[],C536847,Familial Multiple Trichodiscomas,[],NaN,NaN


#### Drug

In [4]:
df = pd.read_csv(os.path.join(drugbank_path, 'full.csv'))

def safe_eval(x):
    if isinstance(x, str):
        try:
            return ast.literal_eval(x)
        except:
            return []
    return x

cols = ['atc_codes', 'interactions']
for col in cols:
    df[col] = df[col].apply(safe_eval)

df['len'] = df['interactions'].apply(lambda x: len(x))

In [5]:
df

,drugbank_id,name,description,atc_codes,interactions,len
0,DB00001,Lepirudin,Lepirudin is a recombinant hirudin formed by 6...,[B01AE02],"[DB06605, DB06695, DB01254, DB01609, DB01586, ...",654
1,DB00002,Cetuximab,Cetuximab is a recombinant chimeric human/mous...,[L01FE01],"[DB00255, DB00269, DB00286, DB00655, DB00783, ...",418
2,DB00003,Dornase alfa,Dornase alfa is a biosynthetic form of human d...,[R05CB13],[],0
3,DB00004,Denileukin diftitox,Denileukin diftitox is an IL2-receptor-directe...,[L01XX29],"[DB00012, DB00016, DB08894, DB09107, DB00281, ...",38
4,DB00005,Etanercept,Dimeric fusion protein consisting of the extra...,[L04AB01],"[DB08879, DB00531, DB06643, DB00065, DB00008, ...",1041
...,...,...,...,...,...,...
19837,DB32266,ARBM101,NaN,[],[],0
19838,DB32267,Sorfequiline,NaN,[],[],0
19839,DB32269,Zamubafusp alfa,Zamubafusp alfa is an immunoglobulin-peptide f...,[],[],0
19840,DB32272,Netanasvir,NaN,[],[],0


In [13]:
snap = pd.read_csv(os.path.join(snap_path, 'snap.tsv'), sep='\t')

In [15]:
snap.columns = ['id1', 'id2']

snap

,id1,id2
0,DB00575,DB00806
1,DB01242,DB08893
2,DB01151,DB08883
3,DB01235,DB01275
4,DB00018,DB00333
...,...,...
48508,DB00542,DB01354
48509,DB00476,DB01239
48510,DB00621,DB01120
48511,DB00808,DB01356


#### UML ID Connection

In [4]:
print_tree(uml_path)

├── AMBIGLUI.RRF
├── AMBIGLUI.ctl
├── AMBIGSUI.RRF
├── AMBIGSUI.ctl
├── MRAUI.RRF
├── MRAUI.ctl
├── MRCOLS.RRF
├── MRCOLS.ctl
├── MRCONSO.RRF
├── MRCONSO.ctl
├── MRCUI.RRF
├── MRCUI.ctl
├── MRDEF.RRF
├── MRDEF.ctl
├── MRDOC.RRF
├── MRDOC.ctl
├── MRFILES.RRF
├── MRFILES.ctl
├── MRHIER.RRF
├── MRHIER.ctl
├── MRHIST.RRF
├── MRHIST.ctl
├── MRMAP.RRF
├── MRMAP.ctl
├── MRRANK.RRF
├── MRRANK.ctl
├── MRREL.RRF
├── MRREL.ctl
├── MRSAB.RRF
├── MRSAB.ctl
├── MRSAT.RRF
├── MRSAT.ctl
├── MRSMAP.RRF
├── MRSMAP.ctl
├── MRSTY.RRF
├── MRSTY.ctl
├── MRXNS_ENG.RRF
├── MRXNS_ENG.ctl
├── MRXNW_ENG.RRF
├── MRXNW_ENG.ctl
├── MRXW_ARA.RRF
├── MRXW_BAQ.RRF
├── MRXW_BAQ.ctl
├── MRXW_CHI.RRF
├── MRXW_CHI.ctl
├── MRXW_CZE.RRF
├── MRXW_CZE.ctl
├── MRXW_DAN.RRF
├── MRXW_DAN.ctl
├── MRXW_DUT.RRF
├── MRXW_DUT.ctl
├── MRXW_ENG.RRF
├── MRXW_ENG.ctl
├── MRXW_EST.RRF
├── MRXW_EST.ctl
├── MRXW_FIN.RRF
├── MRXW_FIN.ctl
├── MRXW_FRE.RRF
├── MRXW_FRE.ctl
├── MRXW_GER.RRF
├── MRXW_GER.ctl
├── MRXW_GRE.RRF
├── MRXW_GRE.ctl
├──

In [19]:
import csv

cols = [
    "CUI","LAT","TS","LUI","STT","SUI","ISPREF",
    "AUI","SAUI","SCUI","SDUI","SAB","TTY",
    "CODE","STR","SRL","SUPPRESS","CVF","EMPTY"
]

df = pd.read_csv(
    os.path.join(uml_path, "MRCONSO.RRF"),
    sep="|",
    names=cols,
    dtype=str,
    engine="python",
    quoting=csv.QUOTE_NONE,
    on_bad_lines="skip"
)

In [20]:
df

,CUI,LAT,TS,LUI,STT,SUI,ISPREF,AUI,SAUI,SCUI,SDUI,SAB,TTY,CODE,STR,SRL,SUPPRESS,CVF,EMPTY
0,C0000005,ENG,P,L0000005,PF,S0007492,Y,A26634265,NaN,M0019694,D012711,MSH,PEP,D012711,(131)I-Macroaggregated Albumin,0,N,NaN,NaN
1,C0000005,ENG,S,L0270109,PF,S0007491,Y,A26634266,NaN,M0019694,D012711,MSH,ET,D012711,(131)I-MAA,0,N,NaN,NaN
2,C0000039,ENG,P,L0000039,PF,S17175117,N,A28315139,9194921,1926948,NaN,RXNORM,IN,1926948,"1,2-dipalmitoylphosphatidylcholine",0,N,NaN,NaN
3,C0000039,ENG,P,L0000039,PF,S17175117,Y,A28572604,NaN,NaN,NaN,MTH,PN,NOCODE,"1,2-dipalmitoylphosphatidylcholine",0,N,NaN,NaN
4,C0000039,ENG,P,L0000039,VC,S0007564,Y,A0016515,NaN,M0023172,D015060,MSH,MH,D015060,"1,2-Dipalmitoylphosphatidylcholine",0,N,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7472744,C6022231,ENG,S,L20220875,PF,S24083083,Y,A37738788,NaN,NaN,NaN,SRC,VAB,V-LNC-TR-TR_281,LNC-TR-TR_281,0,N,NaN,NaN
7472745,C6022232,ENG,P,L20236914,PF,S24083104,Y,A37770208,NaN,NaN,NaN,SRC,VPT,V-LNC-UK-UA_281,"LOINC, Ukrainian, Ukraine Edition, 281",0,N,NaN,NaN
7472746,C6022232,ENG,S,L20231493,PF,S24083084,Y,A37777735,NaN,NaN,NaN,SRC,VAB,V-LNC-UK-UA_281,LNC-UK-UA_281,0,N,NaN,NaN
7472747,C6022233,ENG,P,L20242348,PF,S24083086,Y,A37754479,NaN,NaN,NaN,SRC,VPT,V-LNC-ZH-CN_281,"LOINC, Chinese, China Edition, 281",0,N,NaN,NaN


In [ ]:
mesh_df = df[
    (df["SAB"] == "MSH") &
    (df["LAT"] == "ENG") &
    (df["SUPPRESS"] == "N") &
    (df["CODE"].str.startswith("D", na=False))
][["CUI", "CODE", "STR"]].drop_duplicates()

mesh_df.columns = ["CUI", "MeSH_ID", "MeSH_Name"]

mesh_df

,CUI,MeSH_ID,MeSH_Name
0,C0000005,D012711,(131)I-Macroaggregated Albumin
1,C0000005,D012711,(131)I-MAA
4,C0000039,D015060,"1,2-Dipalmitoylphosphatidylcholine"
5,C0000039,D015060,"1,2 Dipalmitoylphosphatidylcholine"
6,C0000039,D015060,"1,2-Dihexadecyl-sn-Glycerophosphocholine"
...,...,...,...
7448091,C6016647,D000099313,Keratinocyte Carcinoma
7448092,C6016648,D000099317,Hippo Kinases
7448093,C6016648,D000099317,Hippo STK Kinases
7448094,C6016648,D000099317,MST Kinases


In [24]:
rxnorm_df = df[
    (df["SAB"] == "RXNORM") &
    (df["LAT"] == "ENG") &
    (df["SUPPRESS"] == "N") &
    (df["TTY"].isin(["IN", "PIN"]))  # ingredients only
][["CUI", "CODE", "STR"]].drop_duplicates()

rxnorm_df.columns = ["CUI", "RxNorm_ID", "Drug_Name"]

rxnorm_df

,CUI,RxNorm_ID,Drug_Name
2,C0000039,1926948,"1,2-dipalmitoylphosphatidylcholine"
513,C0000294,44,mesna
758,C0000378,1489913,droxidopa
810,C0000392,61,beta-alanine
942,C0000464,73,docosahexaenoate
...,...,...,...
7446130,C6011978,2719298,"4,6-bis(1-methylpentyl)resorcinol"
7446235,C6012063,2719477,linvoseltamab-gcpt
7448958,C6016987,2720359,influenza A virus A/Perth/722/2024 (H3N2) antigen
7449407,C6017377,2721765,eta-tocopherol


In [25]:
mapping = mesh_df.merge(rxnorm_df, on="CUI")

#### SIDER

In [65]:
# nodes = nodes[['uml_id', 'name', 'labels']]

# nodes = nodes.dropna(subset = 'uml_id')

nodes

,uml_id,name,labels
0,C0029089,Ophthalmoplegia,Disease:Test:HPO
4,C0020302,Developmental Glaucoma,Disease:Test:HPO
45,C0085636,Photophobia,Disease:Test:HPO
48,C0026205,Miosis,Disease:Test:HPO
51,C0456909,Blindness,Disease:Test:HPO
...,...,...,...
249086,C0015300,Exophthalmos,Disease:Test:CTD
249127,C0151827,Eye Pain,Disease:Test:CTD
249144,C0015468,Facial Pain,Disease:Test:CTD
249166,C0015523,Factor Xi Deficiency,Disease:Test:CTD


#### CTD


In [ ]:
nodes = pd.read_csv(node_dir)

nodes['type'] = nodes['labels'].apply(lambda x: x.split(':')[0])

base_drug = nodes[nodes['type'] == 'Drug']
base_dis = nodes[nodes['type'] == 'Disease']

In [3]:
chem = os.path.join(ctd_path, 'chemicals.csv')
dis = os.path.join(ctd_path, 'diseases.csv')
chem_dis = os.path.join(ctd_path, 'chem_dis.csv')

# rel = pd.read_csv(chem_dis, comment='#', low_memory=False)
chem_df = pd.read_csv(chem)
# dis_df = pd.read_csv(dis)

In [8]:
chem_df

,name,id,parents,definition,alias
0,(0.017Ferrocene)Amylose,C089250,"['D000075163', 'D000688', 'D005296']",NaN,['(0.017 Ferrocene)Amylose']
1,001-C8-Nbd,C114385,"['D009842', 'D010069']",NaN,"['001 C8 Nbd', 'H-Metyr-Arg-Mearg-D-Leu-Nh(Ch2..."
2,001-C8 Oligopeptide,C114386,['D009842'],NaN,"['001 C8 Oligopeptide', 'H-Metyr-Arg-Mearg-D-L..."
3,"0231A , Streptomyces",C434150,['D006576'],NaN,[]
4,"0231B, Streptomyces",C434149,['D006576'],NaN,[]
...,...,...,...,...,...
179453,"(Z,Z)-6,9-Heneicosadien-11-Ol",C482761,['D000466'],NaN,['Z6Z9-11R-Ol-C21']
179454,"(Z,Z)-6,9-Heptadecadiene",C000712888,['D000466'],NaN,"['6,9-C17']"
179455,"(Z,Z)-7,11-Hexadecadienal",C558532,"['D000466', 'D012724']",NaN,[]
179456,"(Z,Z)-Dodeca-3,6-Dien-1-Ol",C483463,"['D000466', 'D010675']",NaN,[]


#### HPO

In [89]:
hpo = pd.read_csv(os.path.join(hpo_path, 'cleaned.csv'))

In [12]:
hpo.head()

,id,phenotype,description,alternative_name,Is_a,snomed_id,umls_id
0,HP:0000002,Abnormality of body height,Deviation from the norm of height with respect...,[Abnormality of body height],[HP:0001507],[],[C4025901]
1,HP:0000003,Multicystic kidney dysplasia,Multicystic dysplasia of the kidney is charact...,"[Multicystic dysplastic kidney, Multicystic ki...",[HP:0000107],"[204962002, 82525005]",[C3714581]
2,HP:0000005,Mode of inheritance,The pattern in which a particular genetic trai...,[Inheritance],[HP:0000001],[],[C1708511]
3,HP:0000006,Autosomal dominant inheritance,A mode of inheritance that is observed for tra...,"[Autosomal dominant, Autosomal dominant form, ...",[HP:0034345],[263681008],[C0443147]
4,HP:0000007,Autosomal recessive inheritance,A mode of inheritance that is observed for tra...,"[Autosomal recessive, Autosomal recessive form...",[HP:0034345],[258211005],"[C0441748, C4020899]"


In [4]:
list_col = ['alternative_name', 'Is_a', 'snomed_id', 'umls_id']

def parse_list(x):

    # already list or array
    if isinstance(x, (list, np.ndarray)):
        return list(x)

    # missing value
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return []

    # string 
    if isinstance(x, str):
        return ast.literal_eval(x)

    return []

for col in list_col:
    hpo[col] = hpo[col].apply(parse_list)

hpo['description'] = hpo['description'].apply(lambda x: re.sub(r'"\s*\[.*$', '"', str(x)))
hpo['description'] = hpo['description'].apply(lambda x: x.replace('"', ''))

#### DOID

In [9]:
nodes = nodes[['id', 'name']]

In [5]:
do = pd.read_csv(os.path.join(doid_path, 'DOreports', 'DO.csv'))
lookup = pd.read_csv(os.path.join(doid_path, 'DOreports', 'ref.csv'))

In [44]:
do

,id,name,parents,parent_id
0,0040058,"1,4-Phenylenediamine Allergic Contact Dermatitis",Allergic Contact Dermatitis,3042
1,0040069,"1-Chloro-2,4-Dinitrobenzene Allergic Contact D...",Allergic Contact Dermatitis,3042
2,0112248,17-Beta Hydroxysteroid Dehydrogenase 3 Deficiency,Pseudohermaphroditism,3765
3,0040079,"2,4-Dinitrophenyl Allergic Contact Dermatitis",Allergic Contact Dermatitis,3042
4,0111453,2-Aminoadipic 2-Oxoadipic Aciduria,Amino Acid Metabolic Disorder,9252
...,...,...,...,...
12062,10371,Yaws,Primary Bacterial Infectious Disease,NaN
12063,9682,Yellow Fever,Viral Infectious Disease,934
12064,0050468,Yellow Nail Syndrome,Syndrome,225
12065,0060517,Zebrafish Allergy,Fish Allergy,NaN


In [141]:
lookup

,id,name,uml_id,mesh_id,omim_id,doid_id
0,0001816,Angiosarcoma,C0018923,D006394,NaN,DOID:0001816
1,0002116,Pterygium,C0033999,NaN,NaN,DOID:0002116
2,0014667,Disease Of Metabolism,C0025517,D008659,NaN,DOID:0014667
3,0040002,Aspirin Allergy,C0004058,NaN,NaN,DOID:0040002
4,0040003,Benzylpenicillin Allergy,C0571411,NaN,NaN,DOID:0040003
...,...,...,...,...,...,...
11000,9987,Orbit Sarcoma,C1335131,NaN,NaN,DOID:9987
11001,9988,Tertiary Neurosyphilis,C0027927,D009494,NaN,DOID:9988
11002,999,Hypereosinophilic Syndrome,C0014457,D004802,NaN,DOID:999
11003,9993,Hypoglycemia,C0020615,D007003,NaN,DOID:9993


#### SNOMEDCT

In [10]:
print_tree(snomedct_path)

├── Refset
│   ├── Content
│   │   ├── Refset_Simple.csv
│   │   ├── cRefset_Association.csv
│   │   └── cRefset_AttributeValue.csv
│   ├── Language
│   │   └── cRefset_Language.csv
│   ├── Map
│   │   ├── iisssccRefset_ExtendedMap.csv
│   │   └── sRefset_SimpleMap.csv
│   └── Metadata
│       ├── cRefset_MRCMModuleScope.csv
│       ├── cciRefset_RefsetDescriptor.csv
│       ├── ciRefset_DescriptionType.csv
│       ├── cissccRefset_MRCMAttributeDomain.csv
│       ├── scsRefset_ComponentAnnotationStringValue.csv
│       ├── ssRefset_ModuleDependency.csv
│       ├── ssccRefset_MRCMAttributeRange.csv
│       ├── sscsRefset_MemberAnnotationStringValue.csv
│       └── sssssssRefset_MRCMDomain.csv
└── Terminology
    ├── OWL_expression.csv
    ├── concept.csv
    ├── description.csv
    ├── relationship.csv
    ├── relationship_concrete.csv
    ├── stated_relationship.csv
    └── text_definition.csv


In [17]:
df = pd.read_csv(os.path.join(snomedct_path, 'Refset', 'Map', 'iisssccRefset_ExtendedMap.csv'))